## 1️⃣ Google Drive'ı Bağla

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Veri seti yolunuzu buraya yazın (Google Drive'daki konum)
# Örnek: DRIVE_DATA_PATH = '/content/drive/MyDrive/cvpr2017_cvusa'
DRIVE_DATA_PATH = '/content/drive/MyDrive/bitirme_veri/Archive.zip'

# Veri setini Colab'a kopyala (daha hızlı)
!cp -r {DRIVE_DATA_PATH} /content/
print("✓ Veri seti kopyalandı")

In [2]:
!unzip /content/Archive.zip

Archive:  /content/Archive.zip
  End-of-central-directory signature not found.  Either this file is not
  a zipfile, or it constitutes one disk of a multi-part archive.  In the
  latter case the central directory and zipfile comment will be found on
  the last disk(s) of this archive.
unzip:  cannot find zipfile directory in one of /content/Archive.zip or
        /content/Archive.zip.zip, and cannot find /content/Archive.zip.ZIP, period.


## 2️⃣ Orijinal Projeyi Klonla

In [3]:
# Eğer daha önce klonlandıysa sil
!rm -rf University1652-Baseline

# ⚠️ KENDI FORK'UNUZU KULLANIN
# GitHub username'inizi buraya yazın
GITHUB_USERNAME = 'cemresude'  # Örn: 'cemresudeakdag'
BRANCH = 'satellite-drone-rgbd'  # veya 'main'

# Fork'unuzu klonla
!git clone -b {BRANCH} https://github.com/{GITHUB_USERNAME}/University1652-Baseline.git

%cd University1652-Baseline
!ls -la

print(f"✓ Proje klonlandı: {GITHUB_USERNAME}/University1652-Baseline")
print(f"✓ Branch: {BRANCH}")

# Dosyaları kontrol et
import os
files_to_check = ['model_rgbd.py', 'dataset_rgbd.py', 'README_RGBD.md',
                  'train_cvusa.py', 'test_cvusa.py']
for f in files_to_check:
    if os.path.exists(f):
        print(f"✅ {f}")
    else:
        print(f"❌ {f} - EKSIK!")

Cloning into 'University1652-Baseline'...
remote: Enumerating objects: 2037, done.
remote: Counting objects: 100% (435/435), done.
remote: Compressing objects: 100% (156/156), done.
remote: Total 2037 (delta 357), reused 260 (delta 256), pack-reused 1602 (from 1)
Receiving objects: 100% (2037/2037), 130.52 MiB | 34.92 MiB/s, done.
Resolving deltas: 100% (702/702), done.
/content/University1652-Baseline
total 3116
drwxr-xr-x 11 root root    4096 Dec 12 20:00 .
drwxr-xr-x  1 root root    4096 Dec 12 20:00 ..
-rw-r--r--  1 root root   14637 Dec 12 20:00 autoaugment.py
-rw-r--r--  1 root root    1776 Dec 12 20:00 circle_loss.py
-rw-r--r--  1 root root   16060 Dec 12 20:00 COLAB_SATELLITE_DRONE.ipynb
-rw-r--r--  1 root root    4132 Dec 12 20:00 dataset_rgbd.py
-rw-r--r--  1 root root    3880 Dec 12 20:00 demo_4K.py
-rw-r--r--  1 root root    3905 Dec 12 20:00 demo.py
drwxr-xr-x  3 root root    4096 Dec 12 20:00 docs
-rw-r--r--  1 root root    3639 Dec 12 20:00 evaluate_gpu.py
-rw-r--r--  1 

## 3️⃣ Paketleri Yükle

In [4]:
!pip install -q pyyaml scipy matplotlib

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ GPU YOK! Runtime → T4 GPU seçin")

PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4


## 4️⃣ Veri Setini Kontrol Et

In [5]:
# MiDaS Small kurulumu
!pip install -q timm

import torch
import cv2
import numpy as np
from PIL import Image
import os
from tqdm import tqdm

# MiDaS Small modelini yükle
midas = torch.hub.load("intel-isl/MiDaS", "MiDaS_small")
midas_transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
transform = midas_transforms.small_transform

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
midas.to(device)
midas.eval()

print("✅ MiDaS Small yüklendi")

# Satellite görüntüleri için depth map oluştur
def generate_depth_maps(data_dir, output_dir):
    """Satellite görüntüleri için depth map oluştur"""
    os.makedirs(output_dir, exist_ok=True)

    # Train ve test satellite klasörlerini işle
    for split in ['train', 'test']:
        if split == 'train':
            sat_dir = os.path.join(data_dir, split, 'satellite')
        else:
            sat_dir = os.path.join(data_dir, split, 'query_satellite')

        if not os.path.exists(sat_dir):
            continue

        print(f"\n📊 {split} satellite görüntüleri işleniyor...")

        # Her sınıf klasörünü işle
        for class_name in tqdm(os.listdir(sat_dir)):
            class_path = os.path.join(sat_dir, class_name)
            if not os.path.isdir(class_path):
                continue

            output_class_dir = os.path.join(output_dir, split, 'satellite_depth', class_name)
            os.makedirs(output_class_dir, exist_ok=True)

            # Görüntüleri işle
            for img_name in os.listdir(class_path):
                if not img_name.lower().endswith(('.jpg', '.png', '.jpeg')):
                    continue

                img_path = os.path.join(class_path, img_name)
                output_path = os.path.join(output_class_dir, img_name)

                # Zaten işlendiyse atla
                if os.path.exists(output_path):
                    continue

                # Görüntüyü yükle
                img = cv2.imread(img_path)
                img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

                # MiDaS için hazırla
                input_batch = transform(img_rgb).to(device)

                # Depth tahmini
                with torch.no_grad():
                    prediction = midas(input_batch)
                    prediction = torch.nn.functional.interpolate(
                        prediction.unsqueeze(1),
                        size=img_rgb.shape[:2],
                        mode="bicubic",
                        align_corners=False,
                    ).squeeze()

                # Normalize et (0-255)
                depth_map = prediction.cpu().numpy()
                depth_normalized = cv2.normalize(depth_map, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)

                # RGBD oluştur
                img_rgb_pil = Image.fromarray(img_rgb)
                depth_pil = Image.fromarray(depth_normalized)

                # 4 kanallı görüntü oluştur (RGB + D)
                rgbd = np.dstack((img_rgb, depth_normalized))

                # Kaydet (numpy array olarak)
                np.save(output_path.replace('.jpg', '.npy'), rgbd)

                # Görselleştirme için de kaydet
                Image.fromarray(rgbd[:,:,:3]).save(output_path)  # RGB
                depth_pil.save(output_path.replace('.jpg', '_depth.jpg'))  # Depth

    print(f"\n✅ Depth map'ler oluşturuldu: {output_dir}")

# Depth map'leri oluştur
data_dir = '/content/cvpr2017_cvusa'
output_dir = '/content/cvpr2017_cvusa_depth'

generate_depth_maps(data_dir, output_dir)

# Örnek görselleştirme
import matplotlib.pyplot as plt

# İlk örneği göster
sample_class = os.listdir(os.path.join(data_dir, 'train/satellite'))[0]
sample_img = os.listdir(os.path.join(data_dir, f'train/satellite/{sample_class}'))[0]

rgb_path = os.path.join(data_dir, f'train/satellite/{sample_class}/{sample_img}')
depth_path = os.path.join(output_dir, f'train/satellite_depth/{sample_class}/{sample_img.replace(".jpg", "_depth.jpg")}')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))
axes[0].imshow(Image.open(rgb_path))
axes[0].set_title('RGB Satellite')
axes[0].axis('off')

axes[1].imshow(Image.open(depth_path), cmap='plasma')
axes[1].set_title('MiDaS Depth Map')
axes[1].axis('off')
plt.tight_layout()
plt.show()

print("\n📌 Sonraki adım: model.py'yi 4 kanallı giriş için güncelle")

/usr/local/lib/python3.12/dist-packages/torch/hub.py:335: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(


Downloading: "https://github.com/intel-isl/MiDaS/zipball/master" to /root/.cache/torch/hub/master.zip


/usr/local/lib/python3.12/dist-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Loading weights:  None


/usr/local/lib/python3.12/dist-packages/torch/hub.py:335: UserWarning: You are about to download and run code from an untrusted repository. In a future release, this won't be allowed. To add the repository to your trusted list, change the command to {calling_fn}(..., trust_repo=False) and a command prompt will appear asking for an explicit confirmation of trust, or load(..., trust_repo=True), which will assume that the prompt is to be answered with 'yes'. You can also use load(..., trust_repo='check') which will only prompt for confirmation if the repo is not already trusted. This will eventually be the default behaviour
  warnings.warn(


Downloading: "https://github.com/rwightman/gen-efficientnet-pytorch/zipball/master" to /root/.cache/torch/hub/master.zip
Downloading: "https://github.com/rwightman/pytorch-image-models/releases/download/v0.1-weights/tf_efficientnet_lite3-b733e338.pth" to /root/.cache/torch/hub/checkpoints/tf_efficientnet_lite3-b733e338.pth
Downloading: "https://github.com/isl-org/MiDaS/releases/download/v2_1/midas_v21_small_256.pt" to /root/.cache/torch/hub/checkpoints/midas_v21_small_256.pt


100%|██████████| 81.8M/81.8M [00:00<00:00, 317MB/s]
Using cache found in /root/.cache/torch/hub/intel-isl_MiDaS_master


✅ MiDaS Small yüklendi

✅ Depth map'ler oluşturuldu: /content/cvpr2017_cvusa_depth


FileNotFoundError: [Errno 2] No such file or directory: '/content/cvpr2017_cvusa/train/satellite'

## 4.5️⃣ MiDaS Depth Estimation Ekle (Opsiyonel)

In [ ]:
import os

data_dir = '/content/cvpr2017_cvusa'

# Beklenen yapı
required_dirs = [
    f'{data_dir}/train/satellite',
    f'{data_dir}/train/drone',
    f'{data_dir}/test/query_satellite',
    f'{data_dir}/test/gallery_drone'
]

print("📂 Veri yapısı kontrolü:")
for d in required_dirs:
    exists = "✅" if os.path.exists(d) else "❌"
    print(f"{exists} {d}")

# İstatistikler
if os.path.exists(f'{data_dir}/train/satellite'):
    sat_count = len(os.listdir(f'{data_dir}/train/satellite'))
    drone_count = len(os.listdir(f'{data_dir}/train/drone'))
    print(f"\n📊 Train: {sat_count} satellite, {drone_count} drone sınıfı")

## 5️⃣ Scriptleri Otomatik Düzenle (Satellite-Drone)

In [ ]:
print("✅ Fork'unuzdan dosyalar yüklendi - düzenleme gerekmiyor!")
print("\n📄 Mevcut dosyalar:")

import os

# Dosya kontrolü
required_files = {
    'train_cvusa.py': 'Satellite-Drone eğitim scripti',
    'test_cvusa.py': 'Test scripti (query_satellite → gallery_drone)',
    'model_rgbd.py': 'RGBD model (4-kanallı)',
    'dataset_rgbd.py': 'RGBD dataset yükleyici',
    'README_RGBD.md': 'Kullanım kılavuzu'
}

for filename, description in required_files.items():
    if os.path.exists(filename):
        print(f"  ✅ {filename} - {description}")
    else:
        print(f"  ❌ {filename} - EKSIK!")

print("\n📌 Notlar:")
print("  • train_cvusa.py: Zaten satellite+drone için düzenlenmiş")
print("  • test_cvusa.py: query_satellite → gallery_drone yapılandırması")
print("  • model_rgbd.py: MiDaS depth için 4-kanallı model")
print("\n🎉 Direkt Hücre 10'a (Eğitim) geçebilirsiniz!")

## 6️⃣ Eğitimi Başlat

In [ ]:
# Eğitim parametreleri
EXPERIMENT_NAME = 'satellite_drone_colab'
BATCH_SIZE = 16      # T4 GPU için 16, A100 için 32-64
EPOCHS = 60
LEARNING_RATE = 0.01
POOL_TYPE = 'avg'    # avg, max, gem, avg+max
VIEWS = 2            # Satellite + Drone

!python train_cvusa.py \
    --name {EXPERIMENT_NAME} \
    --data_dir /content/cvpr2017_cvusa/train \
    --batchsize {BATCH_SIZE} \
    --lr {LEARNING_RATE} \
    --pool {POOL_TYPE} \
    --views {VIEWS} \
    --gpu_ids 0 \
    --h 384 \
    --w 384 \
    --stride 2 \
    --erasing_p 0.5 \
    --color_jitter

## 7️⃣ Eğitim Grafiği

In [ ]:
from IPython.display import Image, display
import os

graph_path = f'model/{EXPERIMENT_NAME}/train.jpg'
if os.path.exists(graph_path):
    display(Image(graph_path))
else:
    print("📊 Grafik henüz oluşmadı")

## 8️⃣ Test / Değerlendirme

In [ ]:
!python test_cvusa.py \
    --name {EXPERIMENT_NAME} \
    --test_dir /content/cvpr2017_cvusa/test \
    --gpu_ids 0 \
    --which_epoch last

## 9️⃣ Modeli Kaydet (Google Drive)

In [ ]:
# Drive'a kaydet
!mkdir -p /content/drive/MyDrive/University1652_models
!cp -r model/{EXPERIMENT_NAME} /content/drive/MyDrive/University1652_models/

print(f"✅ Model kaydedildi: /content/drive/MyDrive/University1652_models/{EXPERIMENT_NAME}")

# Veya zip'le indir
!cd model && zip -r {EXPERIMENT_NAME}.zip {EXPERIMENT_NAME}
from google.colab import files
files.download(f'model/{EXPERIMENT_NAME}.zip')

## 🔟 Demo

In [ ]:
!python demo.py --name {EXPERIMENT_NAME} --which_epoch last

# Sonucu göster
if os.path.exists('show.png'):
    display(Image('show.png'))
else:
    print("Demo sonucu bulunamadı")

## ⚠️ ALTERNATİF: Manuel Dosya Yükleme

**Eğer Hücre 9'da sorun yaşıyorsanız bu yöntemi kullanın:**
1. Local bilgisayarınızdan düzenlenmiş `train_cvusa.py` ve `test_cvusa.py` dosyalarını yükleyin
2. Hücre 9'u **çalıştırmayın**, direkt Hücre 10'a geçin

In [ ]:
# Düzenlenmiş dosyaları local'den yükle
from google.colab import files
import shutil

print("📤 train_cvusa.py yükleyin...")
uploaded = files.upload()
if 'train_cvusa.py' in uploaded:
    shutil.copy('train_cvusa.py', '/content/University1652-Baseline/train_cvusa.py')
    print("✅ train_cvusa.py kopyalandı")

print("\n📤 test_cvusa.py yükleyin...")
uploaded = files.upload()
if 'test_cvusa.py' in uploaded:
    shutil.copy('test_cvusa.py', '/content/University1652-Baseline/test_cvusa.py')
    print("✅ test_cvusa.py kopyalandı")

print("\n✅ Dosyalar hazır! Hücre 10'a geçin.")